### Running this notebook

**The visualizations in this notebook will run in [jupyter lab](https://github.com/jupyterlab/jupyterlab#installation), not jupyter notebook. Google colab is not supported either. VS Code notebooks _might_ work but that has not been tested.** See the fastplotlib supported frameworks for more info: https://github.com/fastplotlib/fastplotlib/#supported-frameworks 

In [1]:
import os
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import mesmerize_core as mc
import mesmerize_viz
import fastplotlib as fpl
import bruker_concat_tif as ct
from tifffile import TiffWriter

from mesmerize_core.caiman_extensions.cnmf import cnmf_cache

from pathlib import Path

if os.name == "nt":
    # disable the cache on windows, this will be automatic in a future version
    cnmf_cache.set_maxsize(0)

pd.options.display.max_colwidth = 120

## Set paths

In [2]:
# data path
data_path = [Path('D:/Data/2P/VG1GC6_I1M0/12012023/TSeries-12012023-1452-001')]
print(f"Load data from {data_path}.")

# output path
export_path = Path('H:/Analysis/2P/VG1GC6_I1M0/12012023/run1/mesmerize/')

# Create export_path if it doesn't exist
if not os.path.exists(export_path):
    os.makedirs(export_path)
    
mc.set_parent_raw_data_path(export_path) 

# movie path
movie_path = Path.joinpath(export_path, 'cat_tiff_bt.tiff')

# batch path
batch_path = Path.joinpath(export_path, 'batch.pickle')

print(f"Export data to {export_path}.") 
print(f"Movie path: {movie_path}.") 
print(f"Batch path: {batch_path}.") 


Load data from [WindowsPath('D:/Data/2P/VG1GC6_I1M0/12012023/TSeries-12012023-1452-001')].
Export data to H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize.
Movie path: H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize\cat_tiff_bt.tiff.
Batch path: H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize\batch.pickle.


### Concatenate the ome.tif files into a single multi-page tiff file

In [3]:
# Using the concat_tif.py script to concatenate the tif files. If needed, install libtiff with: pip install pylibtiff
ct.concatenate_files(data_path, export_path)

Found 6000 files to concatenate.
Saving concatenated file to directory: H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize
Images are uint16 with range (0, 4095).
Concatenated 6000 files to multi-page TIFF H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize\cat_tiff_mpt.tiff.
Images are uint16 with range 0 - 4095.
Set -r flag to False to disable conversion from uint12 to uint16.
Converted data range: 0 - 65535
Removed H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize\cat_tiff_mpt.tiff
Concatenated 6000 files to BigTIFF H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize\cat_tiff_bt.tiff.
Concatenation completed in 119.78 seconds.


### Create a new batch

This creates a new pandas `DataFrame` with the columns that are necessary for mesmerize. In mesmerize we call this the **batch DataFrame**. You can add additional columns relevant to your experiment, but do not modify columns used by mesmerize.

Note that when you create a DataFrame you will need to use `load_batch()` to load it later. You cannot use `create_batch()` to overwrite an existing batch DataFrame

In [4]:
# create a new batch
df = mc.create_batch(batch_path)
# to load existing batches use `load_batch()`
# df = mc.load_batch(batch_path)
df

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid


### Define parameters and add to batch, algo = mcorr


In [5]:
# copy the mcorr_params2 dict to make some changes
# new_params = deepcopy(mcorr_params2)
default_params = \
{
  'main':
    {
        'strides': [72, 72],
        'overlaps': [36, 36],
        'max_shifts': [20, 20],
        'max_deviation_rigid': 10,
        'border_nan': 'copy',
        'pw_rigid': True,
        'gSig_filt': None
    },
}

# new_params_1 = \
# {
#   'main':
#     {
#         'strides': [18, 18],
#         'overlaps': [24, 24],
#         'max_shifts': [12, 12],
#         'max_deviation_rigid': 6,
#         'border_nan': 'copy',
#         'pw_rigid': True,
#         'gSig_filt': None
#     },
# }

df.caiman.add_item(
  algo='mcorr',
  item_name=movie_path.stem,
  input_movie_path=movie_path,
  params=default_params
)
# df.caiman.add_item(
#   algo='mcorr',
#   item_name=movie_path.stem,
#   input_movie_path=movie_path,
#   params=new_params_1
# )

df


,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
0,mcorr,cat_tiff_bt,cat_tiff_bt.tiff,"{'main': {'strides': (72, 72), 'overlaps': (36, 36), 'max_shifts': (20, 20), 'max_deviation_rigid': 10, 'border_nan'...",None,2023-12-28T17:17:27,None,None,None,c2107b47-3336-487c-9681-ff5c41246496


### Run batch items.

`df.iterrows()` iterates through rows and returns the numerical index and row for each iteration

In [6]:
for i, row in df.iterrows():
    # If already processed, skip
    if row["outputs"] is not None:
        print(f"Skipping batch item {i}, id {row.uuid}, algo {row.algo}. Already run.") 
        continue
    process = row.caiman.run()
    
    # on Windows you MUST reload the batch dataframe after every iteration because it uses the `local` backend.
    # this is unnecessary on Linux & Mac
    # "DummyProcess" is used for local backend so this is automatic
    if process.__class__.__name__ == "DummyProcess":
        df = df.caiman.reload_from_disk()

Running c2107b47-3336-487c-9681-ff5c41246496 with local backend
starting mc
mc finished successfully!
computing projections
Computing correlation image
finished computing correlation image


### Load outputs to dataframe

In [7]:
df = df.caiman.reload_from_disk()
df

,algo,item_name,input_movie_path,params,outputs,added_time,ran_time,algo_duration,comments,uuid
0,mcorr,cat_tiff_bt,cat_tiff_bt.tiff,"{'main': {'strides': (72, 72), 'overlaps': (36, 36), 'max_shifts': (20, 20), 'max_deviation_rigid': 10, 'border_nan'...",{'mean-projection-path': c2107b47-3336-487c-9681-ff5c41246496\c2107b47-3336-487c-9681-ff5c41246496_mean_projection.n...,2023-12-28T17:17:27,2023-12-28T17:32:40,903.0 sec,None,c2107b47-3336-487c-9681-ff5c41246496


### Visualize using fastplotlib


In [8]:
# first item is just the raw movie
movies = [df.iloc[0].caiman.get_input_movie()]

# subplot titles
subplot_names = ["raw"]

# add all the mcorr outputs to the list
for i, row in df.iterrows():
    # Select which row to display
    # if i == 2 or i == 3:
    # add to the list of movies to plot
    movies.append(row.mcorr.get_output())

    # subplot title to show dataframe index
    subplot_names.append(f"ix {i}")

# create the widget
mcorr_iw_multiple = fpl.ImageWidget(
    data=movies,  # list of movies
    window_funcs={"t": (np.mean, 1)}, # window functions as a kwarg, this is what the slider was used for in the ready-made viz
    grid_plot_kwargs={"size": (900, 900)},
    names=subplot_names,  # subplot names used for titles
    cmap="gray" #"gnuplot2"
)

mcorr_iw_multiple.show()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\widgets\image.py:74: UserWarning: Invalid 'window size' value for function: <function mean at 0x00000269AD0AAE60>, setting 'window size' = None for this function. Valid values are integers >= 3.
  warn(


RFBOutputContext()

C:\Users\pseud\miniforge3\envs\mescore\lib\site-packages\fastplotlib\graphics\_features\_base.py:34: UserWarning: converting float64 array to float32
  warn(f"converting {array.dtype} array to float32")


JupyterOutputContext(children=(JupyterWgpuCanvas(css_height='900px', css_width='900px'), IpywidgetToolBar(chil…

In [9]:
mcorr_iw_multiple.close()

#### Clip and save the motion corrected memmap values to the 0-65535 range

In [10]:
# Get the motion corrected output as a memmaped numpy array
mcorr_movie = df.iloc[0].mcorr.get_output() 
# Clip, but don't convert to uint16 yet (caiman expects 32 bit float)
mcorr_movie_16bit = np.clip(mcorr_movie, 0, 2**16-1)

# get path of mcorr obj on disk so we can overwrite
mcorr_path = df.iloc[0].mcorr.get_output_path()
original_mmap_path = Path(mcorr_path)
print(f"mcorr_movie path: {mcorr_path}")

# Create a new memmap file with write access
clipped_mmap_path = original_mmap_path.parent / f"clipped_{original_mmap_path.name}"
clipped_mcorr_movie = np.memmap(clipped_mmap_path, dtype='float32', mode='w+', shape=mcorr_movie_16bit.shape)

# Copy the data from mcorr_movie_16bit to the new memmap file
np.copyto(clipped_mcorr_movie, mcorr_movie_16bit)

# Flush changes to disk and close the memmap and the original memmap
del clipped_mcorr_movie
del mcorr_movie

# Code below does not work on Windows while memmap is open

# Create a backup copy of the original memmap file then delete the original
# original_mmap_path.rename(original_mmap_path.parent / f"original_{original_mmap_path.name}")
# print(f"Original motion corrected movie saved to {original_mmap_path.parent / f'original_{original_mmap_path.name}'}")

# Finally, rename the clipped memmap file to the original name 
# clipped_mmap_path.rename(original_mmap_path)
# Delete the backup copy 
# (original_mmap_path.parent / f"original_{original_mmap_path.name}").unlink()

mcorr_movie path: H:\Analysis\2P\VG1GC6_I1M0\12012023\run1\mesmerize\c2107b47-3336-487c-9681-ff5c41246496\c2107b47-3336-487c-9681-ff5c41246496-cat_tiff_bt_els__d1_1024_d2_1024_d3_1_order_F_frames_6000.mmap


#### Save the motion corrected movie (memmaped array) as a BigTIFF

In [11]:
# Save the motion corrected movie as a uint16 tiff
# mcorr_tif_path = Path.joinpath(export_path, f"mcorr_{movie_path.stem}.tiff")
# with TiffWriter(mcorr_tif_path, bigtiff=True) as tif:
#     tif.write(mcorr_movie_16bit.astype('uint16'), photometric='minisblack')
   
# print(f"Saved motion corrected movie to {mcorr_tif_path}.")

In [ ]:
# Convert values to uint8, and save a 8 bit version
mcorr_tif_path = Path.joinpath(export_path, f"mcorr_u8.tiff")
mcorr_movie_8bit = (mcorr_movie_16bit / (2**16-1) * 255).astype('uint8')
with TiffWriter(mcorr_tif_path, bigtiff=True) as tif:
    tif.write(mcorr_movie_8bit, photometric='minisblack')
   
print(f"Saved 8 bit motion corrected movie to {mcorr_tif_path}.")

#### Create an excerpt of the original movie vs motion corrected movie

In [ ]:
from pathlib import Path
import os
from PIL import Image

# We take the 80 first frames of the original movie and the motion corrected movie, concatenate them horizontally and save the resulting array as a mp4 movie

original_movie = df.iloc[0].caiman.get_input_movie()

# Keep only the first 80 frames of the movies 
original_movie_excerpt = original_movie[:80]
mcorr_movie_excerpt = mcorr_movie_16bit[:80]

### for visualization purposes only ###
# If ranges are significantly different, consider rescaling
original_min, original_max = np.min(original_movie_excerpt), np.max(original_movie_excerpt)
mcorr_min, mcorr_max = np.min(mcorr_movie_excerpt), np.max(mcorr_movie_excerpt)
if original_min != mcorr_min or original_max != mcorr_max:
    normalized_original = (original_movie_excerpt - original_min) / (original_max - original_min) * 65535
    normalized_mcorr = (mcorr_movie_excerpt - mcorr_min) / (mcorr_max - mcorr_min) * 65535
else:
    normalized_original = original_movie_excerpt
    normalized_mcorr = mcorr_movie_excerpt

# Convert to uint16 after normalization
normalized_original_uint16 = normalized_original.astype(np.uint16)
normalized_mcorr_uint16 = normalized_mcorr.astype(np.uint16)

# Concatenate the two movies horizontally
excerpt_movie = np.concatenate((normalized_original_uint16, normalized_mcorr_uint16), axis=2)

# Set the path of the mp4 movie
excerpt_movie_mp4_path = Path.joinpath(export_path, f"excerpt_movie.mp4")
# # Display the first frame of the excerpt_movie 
# first_frame = excerpt_movie[0]
# plt.imshow(first_frame, cmap='gray')
# plt.show()

# Create a directory for temporary frame storage
temp_frame_dir = Path(export_path) / "temp_frames"
os.makedirs(temp_frame_dir, exist_ok=True)

# Save each frame as an image with added padding, to avoid [libx264 @ 0x563ae05f8b40] height not divisible by 2 error
for i, frame in enumerate(excerpt_movie):
    # Add a row of black pixels at the bottom
    padded_frame = np.pad(frame, ((0, 1), (0, 0)), 'constant')
    frame_image = Image.fromarray(padded_frame)
    frame_image.save(temp_frame_dir / f"frame_{i:04d}.png")

# Construct the FFmpeg command to make an MP4 movie with the adjusted size
ffmpeg_command = f"ffmpeg -y -r 30 -f image2 -s 1530x766 -i {temp_frame_dir}/frame_%04d.png -vcodec libx264 -crf 25 -pix_fmt yuv420p {export_path}/excerpt_movie.mp4"
os.system(ffmpeg_command)

# Delete the temporary frame images after creating the video
for file in os.listdir(temp_frame_dir):
    os.remove(temp_frame_dir / file)
os.rmdir(temp_frame_dir)

print(f"Saved excerpt movie to {export_path}/excerpt_movie.mp4")